In [19]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(dotenv_path="../.env", override=True)

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
database = os.getenv('DB_NAME')

print(f"Loaded User: {user}, Port: {port}")

# Explicitly cast port to int and add a fallback if it's missing
port_int = int(port) if port and port != 'None' else 5432

connection_str = f"postgresql://{user}:{password}@{host}:{port_int}/{database}"
engine = create_engine(connection_str)

print("Connected successfully!")

Loaded User: None, Port: None
Connected successfully!


In [23]:
import pandas as pd

df = pd.read_sql("SELECT * FROM diabetes_data", con=engine)
display(df.head())

ProgrammingError: (psycopg2.errors.UndefinedTable) relation "diabetes_data" does not exist
LINE 1: SELECT * FROM diabetes_data
                      ^

[SQL: SELECT * FROM diabetes_data]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [22]:
import pandas as pd

# List all tables accessible in the current database schema
query = """
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public'
"""

df_tables = pd.read_sql(query, con=engine)
display(df_tables)

,table_name


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

target_col = [col for col in df.columns if 'outcome' in col.lower()][0]


X = df.drop(columns=[target_col], errors='ignore')
X = X.select_dtypes(include=['number']).fillna(0)
y = df[target_col].fillna(0)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model training complete!")

Model training complete!


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score


y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1] if len(model.classes_) > 1 else y_pred


print(f"Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_val, y_proba):.4f}\n")
print(classification_report(y_val, y_pred))

Accuracy: 0.8019
ROC-AUC:  0.8801

              precision    recall  f1-score   support

           0       0.79      0.93      0.86       195
           1       0.82      0.58      0.68       113

    accuracy                           0.80       308
   macro avg       0.81      0.76      0.77       308
weighted avg       0.81      0.80      0.79       308



### Model Performance Conclusion

* **Overall Performance:** The baseline Random Forest model achieves a solid overall **Accuracy of 80.19%** and a strong **ROC-AUC of 0.8801**, indicating a good capability to distinguish between diabetic and non-diabetic patients.
* **Class 0 (Non-Diabetic):** The model predicts this class with high reliability, showing a high **recall of 0.93** and an **f1-score of 0.86**, meaning it successfully identifies the vast majority of healthy cases.
* **Class 1 (Diabetic):** While maintaining a respectable **precision of 0.82**, the **recall drops to 0.58**, resulting in an **f1-score of 0.68**. This reveals that the model misses a notable portion of positive diabetes cases (higher false negatives).
* **Next Steps:** To improve clinical utility and catch more positive cases, future iterations should focus on tuning decision thresholds, addressing class imbalance, or experimenting with hyperparameter optimization.